In [1]:
# Load the data
import pandas as pd
import numpy as np

df = pd.read_csv("../data/raw/hillstrom.csv")
print(f"Shape: {df.shape}")
df.head()

Shape: (64000, 13)


,recency,history_segment,history,mens,womens,zip_code,newbie,channel,segment,treatment,visit,conversion,spend
0,10,2) $100 - $200,142.44,1,0,Surburban,0,Phone,Womens E-Mail,1,0,0,0.0
1,6,3) $200 - $350,329.08,1,1,Rural,1,Web,No E-Mail,0,0,0,0.0
2,7,2) $100 - $200,180.65,0,1,Surburban,1,Web,Womens E-Mail,1,0,0,0.0
3,9,5) $500 - $750,675.83,1,0,Rural,1,Web,Mens E-Mail,1,0,0,0.0
4,2,1) $0 - $100,45.34,1,0,Urban,0,Web,Womens E-Mail,1,0,0,0.0


In [2]:
# Null handling
print(df.dtypes)
print(f"\nNull counts:\n{df.isnull().sum()}")
print(f"\nTotal nulls: {df.isnull().sum().sum()}")

recency              int64
history_segment        str
history            float64
mens                 int64
womens               int64
zip_code               str
newbie               int64
channel                str
segment                str
treatment            int64
visit                int64
conversion           int64
spend              float64
dtype: object

Null counts:
recency            0
history_segment    0
history            0
mens               0
womens             0
zip_code           0
newbie             0
channel            0
segment            0
treatment          0
visit              0
conversion         0
spend              0
dtype: int64

Total nulls: 0


In [3]:
# Duplicates
dupes = df.duplicated().sum()
print(f"Exact duplicate rows: {dupes}")

Exact duplicate rows: 6562


In [4]:
''' Treatment group distribution: 
In the Hillstrom dataset, the randomization was roughly 1/3 each: Men's Email, Women's Email, No Email. 
For uplift modeling, you need both a treatment group and a control group. 
You'll later collapse "Mens E-Mail" and "Womens E-Mail" into a single treatment = 1 group, with "No E-Mail" as treatment = 0. The ~1/3 vs ~2/3 split gives you decent statistical power in both groups.
'''
print("Treatment groups:")
print(df["segment"].value_counts())
print(f"\nProportions:\n{df['segment'].value_counts(normalize=True)}")

Treatment groups:
segment
Womens E-Mail    21387
Mens E-Mail      21307
No E-Mail        21306
Name: count, dtype: int64

Proportions:
segment
Womens E-Mail    0.334172
Mens E-Mail      0.332922
No E-Mail        0.332906
Name: proportion, dtype: float64


In [5]:
# Categorical summary
for col in ["history_segment", "zip_code", "channel", "segment"]:
    print(f"\n--- {col} ---")
    print(df[col].value_counts())


--- history_segment ---
history_segment
1) $0 - $100        22970
2) $100 - $200      14254
3) $200 - $350      12289
4) $350 - $500       6409
5) $500 - $750       4911
6) $750 - $1,000     1859
7) $1,000 +          1308
Name: count, dtype: int64

--- zip_code ---
zip_code
Surburban    28776
Urban        25661
Rural         9563
Name: count, dtype: int64

--- channel ---
channel
Web             28217
Phone           28021
Multichannel     7762
Name: count, dtype: int64

--- segment ---
segment
Womens E-Mail    21387
Mens E-Mail      21307
No E-Mail        21306
Name: count, dtype: int64


In [6]:
# Numeric summary
df.describe()

,recency,history,mens,womens,newbie,treatment,visit,conversion,spend
count,64000.000000,64000.000000,64000.000000,64000.000000,64000.000000,64000.000000,64000.000000,64000.000000,64000.000000
mean,5.763734,242.085656,0.551031,0.549719,0.502250,0.667094,0.146781,0.009031,1.050908
std,3.507592,256.158608,0.497393,0.497526,0.499999,0.471257,0.353890,0.094604,15.036448
min,1.000000,29.990000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,2.000000,64.660000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,6.000000,158.110000,1.000000,1.000000,1.000000,1.000000,0.000000,0.000000,0.000000
75%,9.000000,325.657500,1.000000,1.000000,1.000000,1.000000,0.000000,0.000000,0.000000
max,12.000000,3345.930000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,499.000000


In [7]:
# Outcome rates by treatment
print("Visit rate by segment:")
print(df.groupby("segment")["visit"].mean())
print("\nConversion rate by segment:")
print(df.groupby("segment")["conversion"].mean())
print("\nAvg spend by segment:")
print(df.groupby("segment")["spend"].mean())

Visit rate by segment:
segment
Mens E-Mail      0.182757
No E-Mail        0.106167
Womens E-Mail    0.151400
Name: visit, dtype: float64

Conversion rate by segment:
segment
Mens E-Mail      0.012531
No E-Mail        0.005726
Womens E-Mail    0.008837
Name: conversion, dtype: float64

Avg spend by segment:
segment
Mens E-Mail      1.422617
No E-Mail        0.652789
Womens E-Mail    1.077202
Name: spend, dtype: float64
